# Research Notes Repository Chroma Query

In [14]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.research_note_document_loader import ResearchNoteChromaDocumentLoader
from apps.agentic.core.constants import RESEARCH_NOTES_DB_NAME, RESEARCH_NOTES_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = "../../.db"

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [15]:
RESEARCH_NOTES_DB_NAME, RESEARCH_NOTES_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('research_notes',
 'research-notes',
 '../../.db',
 ['research_notes', '.DS_Store', 'pdf_document_library', 'github'])

In [4]:
vs = ResearchNoteChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [5]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['research-notes']
Opened: research-notes
total docs: 51


In [6]:
title = "Thermodynamics"
res = coll.get(where={"title": title})   # no 'include' arg at all
print(f"{title} docs:", len(res["ids"]))

Thermodynamics docs: 51


## Verify Document Metadata

In [7]:
probe = coll.get(limit=5000)  # adjust as needed
metas  = [m for m in probe.get("metadatas", []) if m]
len(metas)

51

In [8]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
print("metadata keys:", sorted(keys))

metadata keys: ['author', 'ext', 'filename', 'h2', 'images', 'path', 'section', 'section_char_offset', 'start_date', 'start_index', 'tags', 'title', 'topic']


In [9]:
print("title counts:", Counter(m.get("title") for m in metas if "title" in m and m.get("title")).most_common(10))
print("author counts:", Counter(m.get("author") for m in metas if "author" in m and m.get("author")).most_common(10))

title counts: [('Thermodynamics', 51)]
author counts: [('Troy Stribling', 51)]


## Search Validation

### Similarity

In [10]:
vs = ResearchNoteChromaDocumentLoader(DB_PATH).vectorstore

docs = vs.similarity_search("Second law of thermodynamics", k=5, filter={"title": "Thermodynamics"})
for d in docs:
    print(d.page_content[:160].replace("\n"," "), "\n")

There are several consistent ways to state the second law of Thermodynamics. The most widehy known but least intuitive is that for any isolated thermodynamic sy 

Another statement of the second law of thermodynamics attributed to cord Kelum is   > A thermodynamic process that extracts heat from a heat reservort and obes  

Since $\Delta S \geqslant 0$ for any process in a isolated system if the entropy if the entropy is maximum the state of the system cannot change its thermodynam 

which would describe the variation in internal energy as the other state variables change. A thermodpinamic process has an intial stale and a final state where  

Dissipative Thermodynamic system ![](https://cdn.mathpix.com/cropped/2025_09_16_4c10231fd464b13e8284g-40.jpg?height=222&width=463&top_left_y=200&top_left_x=638) 



### MMR Search

In [11]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 50, "filter": {"title": "Thermodynamics"}}
)

hits = retriever.invoke("Where are discussions of the Carnot cycle?")
for d in hits[:5]:
    print(d.metadata.get("path"), d.metadata.get("section"), d.metadata.get("section_char_offset"))

./research_notes/thermodynamics-bded6d08-a0fa-4051-84a3-3a2ece187286.md 4 1023
./research_notes/thermodynamics-bded6d08-a0fa-4051-84a3-3a2ece187286.md 16 1220
./research_notes/thermodynamics-bded6d08-a0fa-4051-84a3-3a2ece187286.md 13 233
./research_notes/thermodynamics-bded6d08-a0fa-4051-84a3-3a2ece187286.md 4 2392
./research_notes/thermodynamics-bded6d08-a0fa-4051-84a3-3a2ece187286.md 16 1433


## Search Examples 

In [12]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"title": "Thermodynamics"}, {"author": "Troy Stribling"}]},
    },
)
hits = retriever.invoke("What is a Carnot engine")

[print(h) for h in hits]

page_content='The Carnot Cycle is a Cyclic thermodynamic operating between temperatures $T_{H}$ and $T_{C}$ where $T_{H}>T_{C}$ that transforms neat into work  
To construct a Carot Cycle consider a cyclic thermodynamic process constructed from two isothothemal processes with temperatures $T_{H}$ and $T_{C}$ and two adibatic processes. The first starts at temperature $T_{H}$ and ends at temperature $T_{c}$ and the second starts at temperature $T_{c}$ and ends at temperature $T_{H}$.
An example of this process is shown in the following figure where it is assumed that the thermodynamic system is a flueid which
bas a state function of the  
$$
f(P, V, T)=0
$$  
![](https://cdn.mathpix.com/cropped/2025_09_16_4c10231fd464b13e8284g-30.jpg?height=541&width=732&top_left_y=37&top_left_x=167)' metadata={'start_date': '2018-12-15', 'author': 'Troy Stribling', 'section_char_offset': 1023, 'section': 4, 'title': 'Thermodynamics', 'tags': 'physics', 'topic': 'Review of fundamental thermodynamic cons

[None, None, None, None, None, None, None, None]

### Use MMR Call Directly

In [13]:
docs = vs.max_marginal_relevance_search(
    "Where are discussions of the second law of thermodynamics?",
    k=8,
    fetch_k=60,
    filter={"$and": [{"title": "Thermodynamics"}, {"author": "Troy Stribling"}]},
)

[print(h) for h in docs]

page_content='There are several consistent ways to state the second law of Thermodynamics. The most widehy known but least intuitive is that for any isolated thermodynamic system the entropy always increases. A derivation of this statement is the point of this work. To get there it is useful to consider more intuitive statements that are equivalent. The earliest version of the second Law of Thermodynamics was stated by Clausius,  
A coder object never heats a hotter object.
This is an observation that is widely held and easily understard and serves as an excelent starting point to understand the equivalent statement cloout entropy always increasing.
In this statement cooling an object means transfering energy from the object and heating means transferring energy to the object It is
also understood that a cooler object has a lower temperature than a hotter object.
To dig deeper into this statement the idea of a Heat Reservoir is used. A Heat Reservoir is defined as,  
> A heat reservoir

[None, None, None, None, None, None, None, None]